In [1]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".projectroot").exists())

# UNHCR Displacement : EDA + Preparation

UNHCR's persons-of-concern data records forced displacement: refugees, internally displaced people, and stateless persons, by country and year. Forced displacement gives the risk model a humanitarian signal.

**Input:** `data/raw/unhcr/persons_of_concern.csv` (one row per country-of-origin, country-of-asylum, and year).  
**Output:** `data/interim/unhcr/unhcr_clean.csv` (one row per country-year, keyed on ISO3).

In [2]:
import os
import pandas as pd
import country_converter as coco

RAW = str(ROOT / "data/raw/unhcr/persons_of_concern.csv")
OUT_DIR = str(ROOT / "data/interim/unhcr")
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Setup

In [3]:
u = pd.read_csv(RAW)
for c in ['Refugees', 'IDPs', 'Stateless persons']:
    u[c] = pd.to_numeric(u[c], errors='coerce').fillna(0)
u = u[u['Year'] >= 2015]
print('rows 2015+:', len(u))

rows 2015+: 61147


## 2. Data Discovery

The raw table is keyed by country-of-origin, country-of-asylum, and year, with separate counts for refugees, internally displaced people, and stateless persons. Its shape and a sample are shown below.

In [4]:
print('shape:', u.shape, '| years:', int(u.Year.min()), '-', int(u.Year.max()))
print('columns:', list(u.columns))
u.head()

shape: (61147, 12) | years: 2015 - 2025
columns: ['Year', 'Country of Asylum', 'Country of Origin', 'Country of Asylum ISO', 'Country of Origin ISO', 'Refugees', 'Asylum-seekers', 'IDPs', 'Other people in need of international protection', 'Stateless persons', 'Host community', 'Others of concern']


,Year,Country of Asylum,Country of Origin,Country of Asylum ISO,Country of Origin ISO,Refugees,Asylum-seekers,IDPs,Other people in need of international protection,Stateless persons,Host community,Others of concern
0,2015,Afghanistan,Afghanistan,AFG,AFG,0,0,1174306,0,0,0,150317
1,2015,Egypt,Afghanistan,EGY,AFG,0,33,0,0,0,0,0
2,2015,Argentina,Afghanistan,ARG,AFG,5,0,0,0,0,0,0
3,2015,Australia,Afghanistan,AUS,AFG,7785,1422,0,0,0,0,0
4,2015,Austria,Afghanistan,AUT,AFG,17458,24267,0,0,0,0,0


## 3. Data Preparation

The raw flows are aggregated to one row per country-year: refugees and internally displaced people by country of origin, refugees hosted and stateless persons by country of asylum, then merged into the four features.

In [5]:
origin = (u.groupby(['Country of Origin ISO', 'Year'])
            .agg(refugees_origin=('Refugees', 'sum'), idps=('IDPs', 'sum'))
            .reset_index().rename(columns={'Country of Origin ISO': 'iso3', 'Year': 'year'}))

asylum = (u.groupby(['Country of Asylum ISO', 'Year'])
            .agg(refugees_hosted=('Refugees', 'sum'), stateless=('Stateless persons', 'sum'))
            .reset_index().rename(columns={'Country of Asylum ISO': 'iso3', 'Year': 'year'}))

df = origin.merge(asylum, on=['iso3', 'year'], how='outer')
for c in ['refugees_origin', 'idps', 'refugees_hosted', 'stateless']:
    df[c] = df[c].fillna(0)
_valid = coco.convert(df['iso3'].astype(str).tolist(), to='ISO3', not_found='ZZZ')
df = df[[v != 'ZZZ' for v in _valid]]
assert df.duplicated(['iso3', 'year']).sum() == 0, 'duplicate iso3-year keys!'
df = df[['iso3', 'year', 'refugees_origin', 'idps', 'refugees_hosted', 'stateless']]
print('aggregated:', df.shape, '| countries:', df['iso3'].nunique(), '| years:', df['year'].min(), '-', df['year'].max())

TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
TIB not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
UNK not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3
XXA not found in ISO3


aggregated: (2236, 6) | countries: 211 | years: 2015 - 2025


In [6]:
print('top refugee-origin in 2022 (crisis list):')
print(df[df['year'] == 2022].nlargest(6, 'refugees_origin').to_string(index=False))
print('\nfeature ranges (heavily skewed counts):')
feat = ['refugees_origin', 'idps', 'refugees_hosted', 'stateless']
print(df[feat].describe().T[['50%', 'max']].round(0).to_string())

top refugee-origin in 2022 (crisis list):
iso3  year  refugees_origin      idps  refugees_hosted  stateless
 SYR  2022        6559736.0 6780994.0          13121.0   160000.0
 UKR  2022        5684177.0 5914000.0           2520.0    36233.0
 AFG  2022        5661717.0 3254002.0          52159.0        0.0
 SSD  2022        2295082.0 1474679.0         308369.0    10500.0
 MMR  2022        1251618.0 1504848.0              0.0   630000.0
 COD  2022         932680.0 5541021.0         520544.0        0.0

feature ranges (heavily skewed counts):
                    50%         max
refugees_origin   756.0   6848865.0
idps                0.0  11559970.0
refugees_hosted  2393.0   3764517.0
stateless           0.0   1143096.0


The result is a per-country-year table with the four displacement features (refugees_origin, idps, refugees_hosted, stateless), about 2,236 rows. The 2022 top-origin list matches the known crises (Syria, Ukraine, Afghanistan, South Sudan).

## 4. Exploratory Data Analysis

I set a consistent chart style, then load the GPR target, average it to a yearly value per country, join it to the displacement features, and rank them by Spearman correlation with GPR.

In [7]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':200,'font.family':'DejaVu Sans','font.size':11,
    'axes.titlesize':14,'axes.titleweight':'bold','axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.color':'#E9E9E9','legend.frameon':False})
NAVY, BLUE, RED = '#1F3864', '#2E75B6', '#C0392B'
feat = ['refugees_origin','idps','refugees_hosted','stateless']
gpr = pd.read_csv(str(ROOT / "data/interim/gpr/gpr_monthly.csv"))
gpr['year'] = pd.to_datetime(gpr['month']).dt.year
target = gpr[(gpr.year>=2015)&(gpr.year<=2023)].groupby(['iso3','year'])['gpr'].mean().reset_index()
ms = df.merge(target, on=['iso3','year'], how='inner')
print('labeled country-years:', len(ms), '| countries:', ms.iso3.nunique())
ms[feat+['gpr']].corr(method='spearman')['gpr'].drop('gpr').sort_values(ascending=False).to_frame('spearman_with_gpr').round(3)

labeled country-years: 387 | countries: 43


,spearman_with_gpr
refugees_hosted,0.453
stateless,0.191
refugees_origin,0.068
idps,-0.081


Across 387 labeled country-years (43 countries), refugees_hosted is the strongest GPR correlate (0.45). Producing refugees is weak (refugees_origin 0.07) and IDPs slightly negative (-0.08).

I visualized the top 15 countries by a chosen displacement metric, for a specific year or Overall.

In [8]:
from ipywidgets import interact, Dropdown, Layout
YEARS = ['Overall'] + sorted(df.year.unique().tolist())
FEATS = ['refugees_origin','idps','refugees_hosted','stateless']
WSTYLE = {'description_width':'90px'}; WIDE = Layout(width='430px')

def top_origin(metric='refugees_origin', year='Overall'):
    if year == 'Overall':
        s = df.groupby('iso3')[metric].mean(); tag = 'overall mean'
    else:
        s = df[df.year == int(year)].groupby('iso3')[metric].sum(); tag = str(year)
    top = s.nlargest(15)[::-1]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top.index, top.values/1e6, color=RED, edgecolor='white', linewidth=0.5, zorder=3)
    ax.set_xlabel(f'{metric} (millions)'); ax.set_title(f'Top 15 by {metric} ({tag})', pad=12)
    ax.grid(axis='y', visible=False); fig.tight_layout(); plt.show()

interact(top_origin,
         metric=Dropdown(options=FEATS, value='refugees_origin', description='Metric', style=WSTYLE, layout=WIDE),
         year=Dropdown(options=YEARS, value='Overall', description='Year', style=WSTYLE, layout=WIDE))

interactive(children=(Dropdown(description='Metric', layout=Layout(width='430px'), options=('refugees_origin',…

<function __main__.top_origin(metric='refugees_origin', year='Overall')>

The top refugee-origin countries are Syria, Afghanistan, South Sudan and Ukraine, Sudan and Myanmar. The feature lands on the right countries.

I conducted a redundancy check: Spearman correlation among the 4 features, to confirm each adds distinct signal (none above 0.90).

In [9]:
df[feat].corr(method='spearman').round(2)

,refugees_origin,idps,refugees_hosted,stateless
refugees_origin,1.00,0.52,0.38,0.05
idps,0.52,1.00,0.24,-0.01
refugees_hosted,0.38,0.24,1.00,0.31
stateless,0.05,-0.01,0.31,1.00


The strongest pair is refugees_origin with IDPs at 0.52 (both conflict-driven). So the four capture genuinely distinct facets, producing refugees, internal displacement, hosting refugees, and statelessness, and all are kept.

## 5. Validate and Save

In [10]:
out_path = f'{OUT_DIR}/unhcr_clean.csv'
df.to_csv(out_path, index=False)
print('Saved:', out_path, '|', df.shape[0], 'rows x', df.shape[1], 'cols')
print('NOTE: fill 0 (not NaN) for absent country-years on join.')

Saved: /Users/oussamaennaciri/Documents/Education/Academique/Regis/MSDS692 S40 Data Science Practicum/data/interim/unhcr/unhcr_clean.csv | 2236 rows x 6 cols
NOTE: fill 0 (not NaN) for absent country-years on join.


Claude (Anthropic) was used only as a collaborator for writing code and polishing the notebooks. All analytical decisions, interpretations, and research were conducted independently by me.